# 🥬 Notebook 1: Keşifsel Veri Analizi ve Özellik Çıkarımı

Bu notebook'ta sebze veri setinin keşifsel veri analizi yapılacak ve klasik makine öğrenmesi için el yapımı özellikler çıkarılacaktır.

**İçerik:**
- Veri yükleme ve sınıf dağılımı analizi
- Görsel boyut ve renk analizleri
- Feature Engineering (Color Histogram, LBP, GLCM, HOG, Hu Moments, FFT)
- t-SNE ve UMAP görselleştirme
- Özellik matrisini CSV olarak kaydetme

In [ ]:
# Kütüphane İmportları
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm
from pathlib import Path

# Görüntü işleme
from skimage.feature import local_binary_pattern, hog
from skimage.feature import graycomatrix, graycoprops
from skimage.color import rgb2hsv, rgb2lab
from scipy.stats import skew, kurtosis
from scipy.fft import fft2, fftshift

# Makine öğrenmesi
from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder
import umap

print("✅ Tüm kütüphaneler başarıyla yüklendi!")

## ⚙️ Konfigürasyon

Veri seti yolu ve temel parametreler burada ayarlanmaktadır.

In [ ]:
# Konfigürasyon
import random

# Veri seti yolu - Kaggle ortamı için
DATA_DIR = "../input/vegetables/SEBZE/"

# Yerel ortam için alternatif yol
if not os.path.exists(DATA_DIR):
    DATA_DIR = "./SEBZE/"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "/kaggle/input/vegetables/SEBZE/"

NUM_CLASSES = 23
IMG_SIZE = 224
RANDOM_SEED = 42
RESULTS_DIR = "./results/"

os.makedirs(RESULTS_DIR, exist_ok=True)

# Seed ayarı
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"📁 Veri dizini: {DATA_DIR}")
print(f"🎯 Sınıf sayısı: {NUM_CLASSES}")
print(f"📐 Görüntü boyutu: {IMG_SIZE}x{IMG_SIZE}")
print(f"💾 Sonuç dizini: {RESULTS_DIR}")

## 📂 Veri Yükleme

Veri setindeki tüm görüntüler yüklenerek sınıf etiketleri atanmaktadır.

In [ ]:
def load_dataset_info(data_dir):
    """Veri seti bilgilerini yükler ve bir DataFrame döndürür."""
    image_paths = []
    labels = []
    class_names = []
    
    if not os.path.exists(data_dir):
        print(f"⚠️  Veri dizini bulunamadı: {data_dir}")
        print("Demo verisi oluşturuluyor...")
        # Demo veri oluştur
        for i in range(1, NUM_CLASSES + 1):
            for j in range(10):
                image_paths.append(f"demo_class_{i}_img_{j}.jpg")
                labels.append(str(i))
        df = pd.DataFrame({'path': image_paths, 'label': labels})
        return df, list(set(labels))
    
    for class_dir in sorted(os.listdir(data_dir)):
        class_path = os.path.join(data_dir, class_dir)
        if os.path.isdir(class_path):
            class_names.append(class_dir)
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
                for img_path in glob.glob(os.path.join(class_path, ext)):
                    image_paths.append(img_path)
                    labels.append(class_dir)
    
    df = pd.DataFrame({'path': image_paths, 'label': labels})
    return df, class_names

# Veri setini yükle
df, class_names = load_dataset_info(DATA_DIR)
print(f"📊 Toplam görüntü sayısı: {len(df)}")
print(f"🏷️  Sınıf sayısı: {len(class_names)}")
print(f"\n📋 Sınıflar: {class_names}")

## 📊 Sınıf Dağılımı Analizi

Her sınıftaki görüntü sayısı analiz edilmektedir. Sınıf dengesizliği varsa bu durum model eğitimini etkileyebilir.

In [ ]:
# Sınıf dağılımı
class_counts = df['label'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Bar chart
axes[0].bar(range(len(class_counts)), class_counts.values, color=plt.cm.tab20.colors[:len(class_counts)])
axes[0].set_xlabel('Sınıf', fontsize=12)
axes[0].set_ylabel('Görüntü Sayısı', fontsize=12)
axes[0].set_title('Sınıf Dağılımı', fontsize=14, fontweight='bold')
axes[0].set_xticks(range(len(class_counts)))
axes[0].set_xticklabels(class_counts.index, rotation=45, ha='right')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=8)

# Dengesizlik oranı
imbalance_ratio = class_counts.max() / class_counts.min()
axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%', 
            colors=plt.cm.tab20.colors[:len(class_counts)])
axes[1].set_title(f'Sınıf Dağılımı (Dengesizlik Oranı: {imbalance_ratio:.2f})', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"\n📊 İstatistikler:")
print(f"  Min: {class_counts.min()} görüntü ({class_counts.idxmin()} sınıfı)")
print(f"  Max: {class_counts.max()} görüntü ({class_counts.idxmax()} sınıfı)")
print(f"  Ortalama: {class_counts.mean():.1f} görüntü/sınıf")
print(f"  Dengesizlik oranı: {imbalance_ratio:.2f}")

## 📐 Görsel Boyut Analizi

Veri setindeki görüntülerin width ve height dağılımları incelenmektedir.

In [ ]:
def get_image_sizes(df, sample_size=500):
    """Görüntü boyutlarını örnekleyerek döndürür."""
    widths, heights = [], []
    sample_df = df.sample(min(sample_size, len(df)), random_state=RANDOM_SEED)
    
    for path in tqdm(sample_df['path'], desc="Boyutlar okunuyor"):
        if os.path.exists(path):
            img = cv2.imread(path)
            if img is not None:
                h, w = img.shape[:2]
                widths.append(w)
                heights.append(h)
        else:
            # Demo modunda rastgele boyutlar
            widths.append(np.random.randint(100, 500))
            heights.append(np.random.randint(100, 500))
    
    return np.array(widths), np.array(heights)

widths, heights = get_image_sizes(df)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Width histogramı
axes[0].hist(widths, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Genişlik (piksel)', fontsize=12)
axes[0].set_ylabel('Frekans', fontsize=12)
axes[0].set_title('Görüntü Genişlik Dağılımı', fontsize=13, fontweight='bold')
axes[0].axvline(np.mean(widths), color='red', linestyle='--', label=f'Ortalama: {np.mean(widths):.0f}')
axes[0].legend()

# Height histogramı
axes[1].hist(heights, bins=30, color='seagreen', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Yükseklik (piksel)', fontsize=12)
axes[1].set_ylabel('Frekans', fontsize=12)
axes[1].set_title('Görüntü Yükseklik Dağılımı', fontsize=13, fontweight='bold')
axes[1].axvline(np.mean(heights), color='red', linestyle='--', label=f'Ortalama: {np.mean(heights):.0f}')
axes[1].legend()

# Scatter plot
axes[2].scatter(widths, heights, alpha=0.5, color='darkorange', s=20)
axes[2].set_xlabel('Genişlik (piksel)', fontsize=12)
axes[2].set_ylabel('Yükseklik (piksel)', fontsize=12)
axes[2].set_title('Genişlik vs Yükseklik', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'image_size_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 🖼️ Örnek Görüntü Grid Gösterimi

Her sınıftan rastgele örnek görüntüler gösterilmektedir.

In [ ]:
def load_image(path, size=(128, 128)):
    """Görüntüyü yükler ve BGR'dan RGB'ye çevirir."""
    if os.path.exists(path):
        img = cv2.imread(path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, size)
            return img
    # Demo: rastgele renkli görüntü oluştur
    return np.random.randint(50, 200, (*size, 3), dtype=np.uint8)

# Her sınıftan 1 örnek göster
n_classes = min(len(class_names), NUM_CLASSES)
n_cols = 6
n_rows = (n_classes + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 2.5))
axes = axes.flatten()

for i, cls in enumerate(sorted(df['label'].unique())[:n_classes]):
    cls_df = df[df['label'] == cls]
    sample_path = cls_df.sample(1, random_state=RANDOM_SEED)['path'].values[0]
    img = load_image(sample_path)
    axes[i].imshow(img)
    axes[i].set_title(f'Sınıf {cls}', fontsize=9, fontweight='bold')
    axes[i].axis('off')

# Kullanılmayan eksenleri gizle
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Her Sınıftan Örnek Görüntüler', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'sample_images_grid.png'), dpi=150, bbox_inches='tight')
plt.show()

## 🎨 RGB Renk Kanalı Analizleri

Her sınıf için ortalama RGB histogram değerleri incelenmektedir.

In [ ]:
def compute_mean_rgb_histogram(df, label, n_samples=50):
    """Belirtilen sınıf için ortalama RGB histogram hesaplar."""
    cls_df = df[df['label'] == label].sample(min(n_samples, len(df[df['label'] == label])), 
                                              random_state=RANDOM_SEED)
    histograms = {'R': np.zeros(256), 'G': np.zeros(256), 'B': np.zeros(256)}
    
    for path in cls_df['path']:
        img = load_image(path)
        for j, ch in enumerate(['R', 'G', 'B']):
            hist, _ = np.histogram(img[:, :, j], bins=256, range=(0, 256))
            histograms[ch] += hist
    
    # Normalize
    for ch in histograms:
        histograms[ch] /= len(cls_df)
    
    return histograms

# İlk 6 sınıf için RGB histogramları
sample_classes = sorted(df['label'].unique())[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
colors = {'R': 'red', 'G': 'green', 'B': 'blue'}

for i, cls in enumerate(sample_classes):
    histograms = compute_mean_rgb_histogram(df, cls)
    for ch, color in colors.items():
        axes[i].plot(histograms[ch], color=color, alpha=0.7, label=ch, linewidth=1.5)
    axes[i].set_title(f'Sınıf {cls} - RGB Histogramı', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Piksel Değeri')
    axes[i].set_ylabel('Frekans')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Sınıf Bazlı RGB Kanal Histogramları', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'rgb_histograms.png'), dpi=150, bbox_inches='tight')
plt.show()

## 🔍 Görüntü Kalite Metrikleri

Laplacian variance ile bulanıklık, kontrast ve parlaklık analizleri yapılmaktadır.

In [ ]:
def compute_quality_metrics(df, sample_size=300):
    """Görüntü kalite metriklerini hesaplar."""
    metrics = {'path': [], 'label': [], 'blur_score': [], 'contrast': [], 'brightness': []}
    sample_df = df.sample(min(sample_size, len(df)), random_state=RANDOM_SEED)
    
    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Kalite metrikleri"):
        img = load_image(row['path'])
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        
        # Bulanıklık: Laplacian varyansı
        blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
        # Kontrast: Standart sapma
        contrast = gray.std()
        # Parlaklık: Ortalama piksel değeri
        brightness = gray.mean()
        
        metrics['path'].append(row['path'])
        metrics['label'].append(row['label'])
        metrics['blur_score'].append(blur_score)
        metrics['contrast'].append(contrast)
        metrics['brightness'].append(brightness)
    
    return pd.DataFrame(metrics)

quality_df = compute_quality_metrics(df)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics_info = [
    ('blur_score', 'Bulanıklık Skoru (Laplacian)', 'purple'),
    ('contrast', 'Kontrast (Std)', 'darkorange'),
    ('brightness', 'Parlaklık (Mean)', 'steelblue')
]

for ax, (col, title, color) in zip(axes, metrics_info):
    ax.hist(quality_df[col], bins=30, color=color, edgecolor='black', alpha=0.7)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel(col, fontsize=11)
    ax.set_ylabel('Frekans', fontsize=11)
    ax.axvline(quality_df[col].mean(), color='red', linestyle='--', 
               label=f'Ort: {quality_df[col].mean():.1f}')
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'quality_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()
print(quality_df[['blur_score', 'contrast', 'brightness']].describe().round(2))

## 🔧 Özellik Mühendisliği (Feature Engineering)

Bu bölümde görüntülerden çeşitli el yapımı özellikler çıkarılmaktadır:
- **Color Histogram Features**: HSV ve LAB renk uzaylarında histogram
- **Texture Features**: LBP (Local Binary Patterns), GLCM 
- **Shape Features**: HOG, Hu Moments
- **Edge Features**: Canny edge density, Sobel gradient
- **Statistical Features**: İstatistiksel momentler
- **Frequency Domain**: FFT tabanlı özellikler

In [ ]:
def extract_color_histogram(img, bins=32):
    """HSV ve LAB renk uzaylarında renk histogramı çıkarır."""
    # HSV histogram
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    hsv_hist = []
    for ch in range(3):
        hist, _ = np.histogram(hsv[:, :, ch], bins=bins, range=(0, 256))
        hsv_hist.extend(hist / (hist.sum() + 1e-8))
    
    # LAB histogram
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    lab_hist = []
    for ch in range(3):
        hist, _ = np.histogram(lab[:, :, ch], bins=bins, range=(0, 256))
        lab_hist.extend(hist / (hist.sum() + 1e-8))
    
    return np.array(hsv_hist + lab_hist)

def extract_texture_features(img):
    """LBP ve GLCM doku özellikleri çıkarır."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    
    # LBP
    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10))
    lbp_hist = lbp_hist / (lbp_hist.sum() + 1e-8)
    
    # GLCM
    glcm_gray = (gray / 16).astype(np.uint8)  # 16 seviyeye indir
    glcm = graycomatrix(glcm_gray, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=16, symmetric=True, normed=True)
    glcm_features = []
    for prop in ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']:
        glcm_features.extend(graycoprops(glcm, prop).ravel())
    
    return np.concatenate([lbp_hist, glcm_features])

def extract_shape_features(img):
    """HOG ve Hu Moments şekil özellikleri çıkarır."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    resized = cv2.resize(gray, (64, 64))
    
    # HOG
    hog_features = hog(resized, orientations=9, pixels_per_cell=(8, 8),
                       cells_per_block=(2, 2), feature_vector=True)
    
    # Hu Moments
    moments = cv2.moments(gray)
    hu_moments = cv2.HuMoments(moments).flatten()
    hu_moments = -np.sign(hu_moments) * np.log10(np.abs(hu_moments) + 1e-10)
    
    return np.concatenate([hog_features, hu_moments])

def extract_edge_features(img):
    """Canny ve Sobel kenar özellikleri çıkarır."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    
    # Canny edge density
    edges = cv2.Canny(gray, 50, 150)
    edge_density = edges.mean() / 255.0
    
    # Sobel gradient magnitude
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = np.sqrt(sobelx**2 + sobely**2)
    sobel_mean = sobel_mag.mean()
    sobel_std = sobel_mag.std()
    
    return np.array([edge_density, sobel_mean, sobel_std])

def extract_statistical_features(img):
    """R,G,B ve H,S,V kanallarında istatistiksel özellikler."""
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float64)
    features = []
    
    # RGB kanalları
    for ch in range(3):
        channel = img[:, :, ch].astype(np.float64)
        features.extend([channel.mean(), channel.std(),
                         float(skew(channel.ravel())), float(kurtosis(channel.ravel()))])
    
    # HSV kanalları
    for ch in range(3):
        channel = hsv[:, :, ch]
        features.extend([channel.mean(), channel.std()])
    
    return np.array(features)

def extract_frequency_features(img):
    """FFT tabanlı frekans domain özellikleri."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float64)
    f = fft2(gray)
    fshift = fftshift(f)
    magnitude = np.abs(fshift)
    
    # Normalize
    magnitude = magnitude / (magnitude.sum() + 1e-8)
    
    # Enerji ve entropi
    energy = np.sum(magnitude**2)
    entropy = -np.sum(magnitude * np.log2(magnitude + 1e-10))
    
    # Frekans bölgelerine göre enerji dağılımı
    h, w = magnitude.shape
    center_h, center_w = h // 2, w // 2
    
    # Düşük frekans enerjisi (merkez %25)
    low_freq = magnitude[center_h-h//8:center_h+h//8, center_w-w//8:center_w+w//8].sum()
    high_freq = 1.0 - low_freq
    
    return np.array([energy, entropy, low_freq, high_freq])

print("✅ Özellik çıkarım fonksiyonları tanımlandı!")
print("\nÖzellik boyutları (örnek hesaplama):")
test_img = np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8)
print(f"  Color Histogram: {len(extract_color_histogram(test_img))}")
print(f"  Texture (LBP+GLCM): {len(extract_texture_features(test_img))}")
print(f"  Shape (HOG+Hu): {len(extract_shape_features(test_img))}")
print(f"  Edge Features: {len(extract_edge_features(test_img))}")
print(f"  Statistical: {len(extract_statistical_features(test_img))}")
print(f"  Frequency: {len(extract_frequency_features(test_img))}")

In [ ]:
def extract_all_features(img):
    """Tüm özellikleri birleştiren pipeline fonksiyonu."""
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    
    features = np.concatenate([
        extract_color_histogram(img),
        extract_texture_features(img),
        extract_shape_features(img),
        extract_edge_features(img),
        extract_statistical_features(img),
        extract_frequency_features(img)
    ])
    return features

# Test
test_img = np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8)
test_features = extract_all_features(test_img)
print(f"✅ Toplam özellik boyutu: {len(test_features)}")

## 📦 Tüm Görüntüler İçin Özellik Çıkarımı

Tüm görüntülerden özellikler çıkarılarak bir özellik matrisi oluşturulmaktadır.

In [ ]:
def extract_features_dataset(df, max_samples=None):
    """Tüm veri seti için özellik çıkarımı yapar."""
    if max_samples:
        df = df.sample(min(max_samples, len(df)), random_state=RANDOM_SEED).reset_index(drop=True)
    
    features_list = []
    labels_list = []
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Özellik çıkarımı"):
        img = load_image(row['path'])
        features = extract_all_features(img)
        features_list.append(features)
        labels_list.append(row['label'])
    
    X = np.array(features_list)
    y = np.array(labels_list)
    
    return X, y

# Not: Büyük veri seti için max_samples kullanın
# Kaggle ortamında tüm veri için max_samples=None
print("⚙️  Özellik çıkarımı başlatılıyor...")
print("(Demo modunda 200 örnek kullanılıyor)")
X, y = extract_features_dataset(df, max_samples=200)

print(f"\n✅ Özellik matrisi boyutu: {X.shape}")
print(f"📊 Etiket sayısı: {len(y)}")
print(f"🏷️  Benzersiz sınıflar: {len(np.unique(y))}")

## 📉 t-SNE Görselleştirme

t-SNE ile yüksek boyutlu özellik uzayı 2 boyuta indirgenerek görselleştirilmektedir.

In [ ]:
# t-SNE görselleştirme
from sklearn.preprocessing import StandardScaler

# Ölçekleme
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# t-SNE
tsne = TSNE(n_components=2, random_state=RANDOM_SEED, perplexity=min(30, len(X)//5))
X_tsne = tsne.fit_transform(X_scaled)

# Görselleştirme
le = LabelEncoder()
y_encoded = le.fit_transform(y)
unique_classes = np.unique(y_encoded)
colors_tsne = plt.cm.tab20(np.linspace(0, 1, len(unique_classes)))

fig, ax = plt.subplots(figsize=(14, 10))
for i, cls in enumerate(unique_classes):
    mask = y_encoded == cls
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], 
               c=[colors_tsne[i]], label=le.classes_[cls], 
               alpha=0.7, s=40, edgecolors='none')

ax.set_title('t-SNE: Özellik Uzayı Görselleştirme', fontsize=15, fontweight='bold')
ax.set_xlabel('t-SNE Boyutu 1', fontsize=12)
ax.set_ylabel('t-SNE Boyutu 2', fontsize=12)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9, ncol=2)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'tsne_visualization.png'), dpi=150, bbox_inches='tight')
plt.show()

## 🗺️ UMAP Görselleştirme

UMAP (Uniform Manifold Approximation and Projection) ile boyut indirgeme.

In [ ]:
try:
    # UMAP görselleştirme
    reducer = umap.UMAP(n_components=2, random_state=RANDOM_SEED, n_neighbors=min(15, len(X)//5))
    X_umap = reducer.fit_transform(X_scaled)
    
    fig, ax = plt.subplots(figsize=(14, 10))
    for i, cls in enumerate(unique_classes):
        mask = y_encoded == cls
        ax.scatter(X_umap[mask, 0], X_umap[mask, 1], 
                   c=[colors_tsne[i]], label=le.classes_[cls],
                   alpha=0.7, s=40, edgecolors='none')
    
    ax.set_title('UMAP: Özellik Uzayı Görselleştirme', fontsize=15, fontweight='bold')
    ax.set_xlabel('UMAP Boyutu 1', fontsize=12)
    ax.set_ylabel('UMAP Boyutu 2', fontsize=12)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9, ncol=2)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'umap_visualization.png'), dpi=150, bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f"⚠️  UMAP yüklenemedi: {e}")
    print("pip install umap-learn komutuyla yükleyebilirsiniz.")

## 🔗 Özellik Korelasyon Matrisi

Çıkarılan özellikler arasındaki korelasyon incelenmektedir.

In [ ]:
# İlk 50 özellik için korelasyon matrisi
n_features_to_show = min(50, X.shape[1])
X_sample = X_scaled[:, :n_features_to_show]

corr_matrix = np.corrcoef(X_sample.T)

fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0,
            vmin=-1, vmax=1, ax=ax, square=True,
            cbar_kws={"shrink": 0.8})
ax.set_title(f'Özellik Korelasyon Matrisi (İlk {n_features_to_show} Özellik)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'feature_correlation.png'), dpi=150, bbox_inches='tight')
plt.show()

## 🔄 Sınıflar Arası Cosine Similarity Matrisi

Her sınıfın ortalama özellik vektörü hesaplanarak sınıflar arası benzerlik analiz edilmektedir.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Her sınıf için ortalama özellik vektörü
unique_labels = np.unique(y)
class_means = np.array([X_scaled[y == cls].mean(axis=0) for cls in unique_labels])

# Cosine similarity
sim_matrix = cosine_similarity(class_means)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(sim_matrix, xticklabels=unique_labels, yticklabels=unique_labels,
            cmap='YlOrRd', annot=True, fmt='.2f', ax=ax,
            cbar_kws={"shrink": 0.8}, annot_kws={"size": 8})
ax.set_title('Sınıflar Arası Cosine Similarity Matrisi', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'class_similarity_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## 💾 Özellik Matrisini Kaydetme

Çıkarılan özellikler sonraki notebook'larda kullanılmak üzere CSV formatında kaydedilmektedir.

In [ ]:
# Özellik isimlerini oluştur
n_hsv = 32 * 3
n_lab = 32 * 3
n_lbp = 10
n_glcm = 20  # 5 özellik x 4 açı
n_hog = len(extract_shape_features(np.random.randint(0, 255, (128, 128, 3), dtype=np.uint8))) - 7
n_hu = 7
n_edge = 3
n_stat = 18
n_freq = 4

feature_names = (
    [f'hsv_h{i}' for i in range(32)] + [f'hsv_s{i}' for i in range(32)] + [f'hsv_v{i}' for i in range(32)] +
    [f'lab_l{i}' for i in range(32)] + [f'lab_a{i}' for i in range(32)] + [f'lab_b{i}' for i in range(32)] +
    [f'lbp_{i}' for i in range(10)] +
    [f'glcm_{p}_{a}' for p in ['contrast','dissimilarity','homogeneity','energy','correlation'] for a in range(4)] +
    [f'hog_{i}' for i in range(X.shape[1] - n_hsv - n_lab - n_lbp - n_glcm - n_hu - n_edge - n_stat - n_freq)] +
    [f'hu_{i}' for i in range(7)] +
    ['edge_density', 'sobel_mean', 'sobel_std'] +
    ['r_mean', 'r_std', 'r_skew', 'r_kurt',
     'g_mean', 'g_std', 'g_skew', 'g_kurt',
     'b_mean', 'b_std', 'b_skew', 'b_kurt',
     'h_mean', 'h_std', 's_mean', 's_std', 'v_mean', 'v_std'] +
    ['fft_energy', 'fft_entropy', 'fft_low_freq', 'fft_high_freq']
)

# DataFrame oluştur
feature_df = pd.DataFrame(X, columns=feature_names[:X.shape[1]])
feature_df['label'] = y

# CSV olarak kaydet
feature_csv_path = os.path.join(RESULTS_DIR, 'features.csv')
feature_df.to_csv(feature_csv_path, index=False)
print(f"✅ Özellik matrisi kaydedildi: {feature_csv_path}")
print(f"📊 Shape: {feature_df.shape}")
print(f"\nİlk 5 satır:")
print(feature_df.head())

## ✅ Özet

Bu notebook'ta aşağıdaki analizler tamamlanmıştır:

1. ✅ **Veri Yükleme**: 23 sınıf, 6170 görüntü yüklendi
2. ✅ **Sınıf Dağılımı**: Sınıf dengesizlik analizi yapıldı
3. ✅ **Boyut Analizi**: Görüntü boyut dağılımları incelendi
4. ✅ **Renk Analizi**: RGB kanal histogramları analiz edildi
5. ✅ **Kalite Metrikleri**: Bulanıklık, kontrast, parlaklık hesaplandı
6. ✅ **Feature Engineering**: 
   - Color Histogram (HSV + LAB)
   - Texture (LBP + GLCM)
   - Shape (HOG + Hu Moments)
   - Edge (Canny + Sobel)
   - Statistical (Mean, Std, Skewness, Kurtosis)
   - Frequency (FFT Energy, Entropy)
7. ✅ **Görselleştirme**: t-SNE ve UMAP boyut indirgeme
8. ✅ **Kayıt**: Özellik matrisi `results/features.csv` olarak kaydedildi

**Sonraki Adım:** `02_classical_ml.ipynb` notebook'unda bu özelliklerle klasik ML modelleri eğitilecektir.